In [142]:
# Jupyter Notebook Version of the Resampling Script

# Import necessary libraries
from pathlib import Path
from multiprocessing import Pool
import logging
import pandas as pd
import numpy as np
import SimpleITK as sitk
import click
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact
import os
import nibabel as nib


In [143]:
# Function to visualize CT data interactively
def show_ct_data(ct_data):
    # Ensure ct_data is a NumPy array
    ct_data = np.asarray(ct_data)
    
    # Define the function to plot a specific slice
    def plot_slice(slice_idx):
        plt.figure(figsize=(6, 6))
        plt.imshow(ct_data[:, :, slice_idx], cmap='gray')
        plt.title(f"Slice {slice_idx}")
        # plt.axis('off')
        plt.show()
    
    # Create an interactive slider to select slices
    interact(plot_slice, slice_idx=widgets.IntSlider(min=0, max=ct_data.shape[2] - 1, step=1, value=0))

In [144]:
# Function to visualize CT data with label contours interactively
def show_ct_data_with_labels(ct_data, label_data):
    """
    Display CT image slices with overlaid label contours interactively.

    Parameters:
    - ct_data: NumPy array of CT image data with shape (height, width, slices).
    - label_data: NumPy array of label data with the same shape as ct_data.
    """
    # Ensure ct_data and label_data are NumPy arrays
    ct_data = np.asarray(ct_data)
    label_data = np.asarray(label_data)
    
    # Verify that the shapes match
    if ct_data.shape != label_data.shape:
        raise ValueError("CT data and label data must have the same shape.")
    
    # Define the function to plot a specific slice
    def plot_slice(slice_idx):
        plt.figure(figsize=(6, 6))
        plt.imshow(ct_data[:, :, slice_idx], cmap='gray')
        
        # Overlay the label as a contour
        contours = plt.contour(label_data[:, :, slice_idx], levels=np.unique(label_data), colors='r', linewidths=0.5)
        
        # Optionally, add a colorbar or legend for labels
        # plt.clabel(contours, inline=True, fontsize=8)
        
        plt.title(f"Slice {slice_idx}")
        plt.axis('off')
        plt.show()
    
    # Create an interactive slider to select slices
    interact(plot_slice, slice_idx=widgets.IntSlider(min=0, max=ct_data.shape[2] - 1, step=1, value=0))

In [145]:
def center_crop(arr, size = 128):
    # Calculate the center coordinates
    center_x, center_y = arr.shape[1] // 2, arr.shape[0] // 2

    # Define half the size of the slice (128 // 2 = 64)
    half_size = size // 2

    # Slice the image from the center
    return arr[center_y - half_size:center_y + half_size,
                        center_x - half_size:center_x + half_size,
                        :] 

In [146]:
def separate_labels(mask):
    gtvp = mask[:]
    gtvn = mask[:]

    # 1 = GTVp, 2 = GTVn
    gtvp = np.where(gtvp <= 1, gtvp, 0)
    gtvn = np.where(gtvn>1, gtvn, 0)

    return gtvp

In [147]:
def save_positives_nii(p, ct, pt, lbl):
    # Convert rotated NumPy arrays back to SimpleITK Image objects
    cn_r_img = sitk.GetImageFromArray(np.transpose(ct, (2, 0, 1)))
    pn_r_img = sitk.GetImageFromArray(np.transpose(pt, (2, 0, 1)))
    ln_r_img = sitk.GetImageFromArray(np.transpose(lbl, (2, 0, 1)))

    # save complete images as nii.gz into the folder 

    sitk.WriteImage(cn_r_img, str(output_image_folder / (p + "__CT.nii.gz")))
    sitk.WriteImage(pn_r_img, str(output_image_folder / (p + "__PT.nii.gz")))

    # Save the resampled Label to the output folder
    sitk.WriteImage(ln_r_img, str(output_label_folder / (p + ".nii.gz")))


In [148]:
def save_as_png(pname, ct, pt, lbl):

    assert ct.shape[2] == pt.shape[2]
    assert pt.shape[2] == lbl.shape[2]
    
    for i, s in zip(non_zero_slices, range(ct.shape[2])):

        # print(i, s)
        # plt.imshow(ct_positive[:,:,s])
        name = os.path.join(png_output_image_folder, f'{pname}__CT_{i:02d}.png')
        plt.imsave(name, ct[:,:,s], cmap='gray')

        name = os.path.join(png_output_image_folder, f'{pname}__PT_{i:02d}.png')
        plt.imsave(name, pt[:,:,s], cmap='gray')

        name = os.path.join(png_output_label_folder, f'{pname}_{i:02d}.png')
        plt.imsave(name, lbl[:,:,s], cmap='gray')

    

# start here 
this notebook does the following:
1. load the image 
2. crops the image to 256x256x n
3. finds the slices that has tumors
4. save the result as both ``nii.gz`` and ``.png``

## set the locations

In [149]:
path_input_images = "/Volumes/Seagate/Research/ADDA/resampled_subsets/images"
path_input_labels = "/Volumes/Seagate/Research/ADDA/resampled_subsets/labels"
path_output_images = "/Volumes/Seagate/Research/ADDA/positive_256/images"
path_output_labels = "/Volumes/Seagate/Research/ADDA/positive_256/labels"
png_output_img_path = "/Volumes/Seagate/Research/ADDA/png_256/images"
png_output_label_path = "/Volumes/Seagate/Research/ADDA/png_256/labels"

path_data_breakdown = '/Volumes/Seagate/Research/ADDA/resampled_subsets/dataset.csv'

# Resolve paths to absolute paths
input_image_folder = Path(path_input_images).resolve()
input_label_folder = Path(path_input_labels).resolve()
output_image_folder = Path(path_output_images).resolve()
output_label_folder = Path(path_output_labels).resolve()
png_output_image_folder = Path(png_output_img_path).resolve()
png_output_label_folder = Path(png_output_label_path).resolve()

print('Input image folder:', input_image_folder)
print('Input label folder:', input_label_folder)
print('Output image folder:', output_image_folder)
print('Output label folder:', output_label_folder)
print('png Output label folder:', png_output_image_folder)
print('png Output label folder:', png_output_label_folder)

Input image folder: /Volumes/Seagate/Research/ADDA/resampled_subsets/images
Input label folder: /Volumes/Seagate/Research/ADDA/resampled_subsets/labels
Output image folder: /Volumes/Seagate/Research/ADDA/positive_256/images
Output label folder: /Volumes/Seagate/Research/ADDA/positive_256/labels
png Output label folder: /Volumes/Seagate/Research/ADDA/png_256/images
png Output label folder: /Volumes/Seagate/Research/ADDA/png_256/labels


In [150]:
df = pd.read_csv(path_data_breakdown)
# df = df[['subject_id', 'dataset', 'set']]
df.head()

,subject_id,set,dataset
0,MDA-039,train,source
1,MDA-192,train,source
2,MDA-187,train,source
3,MDA-179,train,source
4,MDA-162,train,source


## processing starts from here

In [152]:
subjects = df['subject_id']

In [156]:
for i, p in enumerate(subjects):
    print(f'working on {p}')
    # 1. load an image 
    # p = 'MDA-162'
    ct = sitk.ReadImage(str([f for f in input_image_folder.rglob(p + "__CT*")][0]))
    pt = sitk.ReadImage(str([f for f in input_image_folder.rglob(p + "__PT*")][0]))


    cn = sitk.GetArrayFromImage(ct)
    pn = sitk.GetArrayFromImage(pt)

    # load label                 
    lbls = [(sitk.ReadImage(str(f)), f.name) for f in input_label_folder.glob(p + "*")]
    ln=[]
    for label, name in lbls:
        ln = sitk.GetArrayFromImage(label) 

    ct_img_cropped = center_crop(cn, size=256)
    pt_img_cropped = center_crop(pn, size=256)
    lbl_cropped = center_crop(ln, size=256)

    # show_ct_data_with_labels(pt_img_cropped, lbl_cropped)

    # 2. keep the gtv only
    lbl_cropped_gtv = separate_labels(lbl_cropped)

    # 3. keep the non-zero slices
    non_zero_slices = [i for i in range(lbl_cropped_gtv.shape[2]) if np.any(lbl_cropped_gtv[:, :, i])]
    if len(non_zero_slices) > 0:
        first_tumor = non_zero_slices[0]
        last_tumor = non_zero_slices[-1]
    else:
        print(f'{p} has no tumors')
        continue

    # print("Indices of slices with non-zero values:", non_zero_slices)
    # print(f'ranging from {first_tumor} to {last_tumor}')

    # 4. remove slices that doesnt have tumor
    ct_positive = ct_img_cropped[:,:,first_tumor:last_tumor+1]
    pt_positive = pt_img_cropped[:,:,first_tumor:last_tumor+1]
    lbl_positive = lbl_cropped_gtv[:,:,first_tumor:last_tumor+1]

    # 5. save cropped images to disk as nii files
    save_positives_nii(p, ct_positive, pt_positive, lbl_positive)

    # 6. save slices as png
    save_as_png(p, ct_positive, pt_positive, lbl_positive)

    # Print counter to show progress every 10 iterations
    if i % 10 == 0:
        percentage = (i / len(subjects)) * 100
        print(f'Round {i}: {percentage:.2f}% completed')

print('finished ...')



working on MDA-039
Round 0: 0.00% completed
working on MDA-192
MDA-192 has no tumors
working on MDA-187
working on MDA-179
MDA-179 has no tumors
working on MDA-162
working on MDA-097
working on MDA-194
working on MDA-095
working on MDA-038
working on MDA-147
working on MDA-150
Round 10: 3.91% completed
working on MDA-133
working on MDA-071
working on MDA-113
working on MDA-023
working on MDA-024
working on MDA-013
working on MDA-059
working on MDA-006
working on MDA-101
working on MDA-065
Round 20: 7.81% completed
working on MDA-120
working on MDA-163
working on MDA-064
working on MDA-025
working on MDA-088
working on MDA-098
working on MDA-099
working on MDA-078
working on MDA-020
working on MDA-100
Round 30: 11.72% completed
working on MDA-021
working on MDA-005
working on MDA-047
working on MDA-173
working on MDA-146
working on MDA-068
working on MDA-053
working on MDA-004
working on MDA-114
working on MDA-055
Round 40: 15.62% completed
working on MDA-061
working on MDA-132
working 

# Sandbox

In [125]:
lbl_positive[:,:,21].max()

2

In [100]:
show_ct_data(ct_positive)

interactive(children=(IntSlider(value=0, description='slice_idx', max=28), Output()), _dom_classes=('widget-in…

In [121]:
non_zero_slices.index(32)

21

In [159]:
show_ct_data_with_labels(pt_img_cropped, lbl_cropped)

interactive(children=(IntSlider(value=0, description='slice_idx', max=297), Output()), _dom_classes=('widget-i…